# GIVE ME A TITLE

This example aims to illustrate the functionality of the `causalprog.graph.ricardo` submodule.
In particular; we will cover the purposes of the major functions within this package, any additional setup the user will need to conduct a-priori, and how to interact with the model that is constructed and estimate the causal bounds.

Our example will be grounded in the framework of continuous treatment models, and we adopt the language and notation of [the description that can be found in the documentation](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/).

## Imports

To keep the actual workflow of the notebook clean, we will conduct all the imports that we require in the cell below.

In [ ]:
import jax
import jax.numpy as jnp

from causalprog.graph.ricardo import (
    MLPAlias,
    ModelParam,
)

jax.config.update("jax_enable_x64", val=True)

<!-- TODO: X-ref to docs pages, also name of submodule shouldn't be ricardo! -->

Within these imports are:

- `jax`, which the package will ultimately use for efficient computations on both CPU and GPU systems. `jax.numpy` is a submodule that provides `numpy`-like operations, and the `jax.config.update` command enables `float64` precision operations as opposed to the default `float32`.
- The `Graph` class from `causalprog.graph` is what `causalprog` uses to describe causal models.
- `causalprog.graph.ricardo` contains helper functions for interacting with continuous treatment models as described in [the documentation](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/).
  We will be discussing these functions in more detail as we encounter them in our workflow.
  The `MLPAlias` and `ModelParam` imports are for type-hinting purposes, and describe the format for MLPs and model parameters that `causalprog` is expecting.
- `causalprog.quadrature` provides quadrature rules for approximating stochastic integrals.
- `causalprog.solvers` provides us with a few JAX-friendly ways to solve the resulting optimisation problem involving the causal response function.

## Example Problem

For this example we will look to solve a problem that reduces to a quadratically-constrained linear objective function, [as described here](https://github-pages.ucl.ac.uk/causalprog/theory/reduce-to-linear-example/).
However we will highlight the points in the workflow that can be - and are expected to be - generalised for actual use-cases.
The example linked to above is just being used in the interest of having a lightweight example that can be run on-demand, and provide an analytic answer.

To this end, we will construct the various mapping functions that this problem entails.

In [ ]:
def mlps_for_example(d_z: int, k_len: int) -> dict[str, MLPAlias]:
    r"""
    Construct MLP-stand-ins used in this problem.

    - $f_m$ returns 0 so that the sigmoid it's passed into always returns 0.5.
    - $f_r$ just returns a 1-vector of appropriate length.
    - $f_{pi}$ also just returns a 1-vector.
    - $g(x, z, l) = -x \mathbb{I}$ where $\mathbb{I}$ is the $\mathbb{R}^{d_z}$ unit
        vector with identical elements.
    - $f_y$ is defined by `f_y`, above.
    """

    def f_m(*args, **kwargs) -> jax.Array:
        """Note that this results in sigmoid(f_m) = 0.5 always."""
        return 0.0

    def f_r(*args, **kwargs) -> jax.Array:
        """Results in the 1-vector in R^d_z after passing through tanh."""
        return jnp.full((d_z,), float("inf"))

    def f_pi(*args, **kwargs) -> jax.Array:
        """Theoretically irrelevant as it will be softmax'd."""
        return jnp.ones((k_len,))

    def g(xzl: dict[str, jax.Array], _: ModelParam) -> jax.Array:
        """Form ensures that m_y^T g gives us a mean of -x/2."""
        return -xzl["x"] * jnp.ones((d_z,)) / jnp.sqrt(d_z)

    def f_y(u_yxl: dict[str, jax.Array], theta_y: ModelParam) -> jax.Array:
        r"""$f_Y(u_y, x, l; \theta_Y) = \frac{\theta_Y}{l}(u_y - x)^2$."""
        return (theta_y / u_yxl["l"]) * (u_yxl["u_y"] - u_yxl["x"]) ** 2

    return {"f_r": f_r, "f_m": f_m, "f_pi": f_pi, "g": g, "f_y": f_y}

Note that in almost all practical examples, the functions created here will not be simple deterministic expressions.

The functions `f_m`, `f_r`, `f_pi`, `f_y` are expected to be multilayer perceptrons as described in [TODO: link Sam's MLP descriptions](FIXME).
The function `g` is expected to be the [inverse map of a pre-trained normalising flow feed-forward network](https://github-pages.ucl.ac.uk/causalprog/theory/continuous-treatments/#model-for-treatment-x).
Though in practice, they can be any `jit`-able function(s) of two arguments;

- The first argument being a dictionary whose keys are the names of the dependent nodes / RVs that the function takes as direct inputs.
  The values of each key should be the corresponding input values for that RV.
- The second argument should be the model parameters $\theta_{\alpha}$ that parametrise the function (or predictive model described by the function).
  This will be a subset of the parameters over which the causal response function will be estimated.

Hence why, in this case, we can simply take them as simple deterministic functions parameterised by their respective parameter values.